In [ ]:
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    RobertaModel,
)
from torch.optim import AdamW
import re
import math

LOGGER = logging.getLogger("technique_tc")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 1. **Data Loading & Preparation**
- Load articles from text files
- Parse task 2 labels (technique classification with span coordinates)
- Handle multi-label spans by expanding to single-label rows
- Extract sentence context for each span

In [ ]:
# =========================
# Data loading & preparation
# =========================

def read_articles_from_folder(folder: str) -> Dict[str, str]:
    """Read all *.txt files and return {article_id: text}.

    Expects filenames like 'article123456789.txt' but only uses the numeric suffix as id.
    """
    folder_path = Path(folder)
    articles: Dict[str, str] = {}
    for file_path in sorted(folder_path.glob("*.txt")):
        name = file_path.stem  # e.g. 'article730081389'
        # Be robust: take everything after 'article' prefix if present, else full stem
        if name.startswith("article"):
            article_id = name[len("article") :]
        else:
            article_id = name
        with file_path.open("r", encoding="utf-8") as handle:
            articles[article_id] = handle.read()
    return articles


def read_task2_labels(path: str) -> Tuple[List[str], List[int], List[int], List[str]]:
    """Read Task 2 (technique classification) labels file.

    Format per line:
        article_id <TAB> technique_label <TAB> span_start <TAB> span_end
    """
    article_ids: List[str] = []
    labels: List[str] = []
    span_starts: List[int] = []
    span_ends: List[int] = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 4:
                continue
            aid, label, start, end = parts
            article_ids.append(aid)
            labels.append(label)
            span_starts.append(int(start))
            span_ends.append(int(end))
    return article_ids, span_starts, span_ends, labels


def load_tc_data(
    articles_folder: str,
    labels_file: str,
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Load articles and Task 2 labels and build a span-level DataFrame.

    Returns a DataFrame with:
        article_id, span_start, span_end, span_text, context, label
    and the mapping article_id -> full article text.
    """
    articles = read_articles_from_folder(articles_folder)
    article_ids, span_starts, span_ends, labels = read_task2_labels(labels_file)

    rows = []
    for aid, s, e, lab in zip(article_ids, span_starts, span_ends, labels):
        text = articles.get(aid, "")
        s = max(0, min(s, len(text)))
        e = max(0, min(e, len(text)))
        span_text = text[s:e]
        # Simple context: a window around the span instead of sentence splitting
        left = max(0, s - 200)
        right = min(len(text), e + 200)
        context = text[left:right].replace("\t", " ").replace("\n", " ")
        rows.append(
            {
                "article_id": aid,
                "span_start": s,
                "span_end": e,
                "span": span_text.replace("\t", " ").replace("\n", " "),
                "context": context,
                "label": lab,
            }
        )
    df = pd.DataFrame(rows)
    return df, articles


def load_tc_test_template(
    articles_folder: str,
    template_labels_file: str,
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Load test articles and template spans.

    template_labels_file has the same format as train labels but label column is a placeholder.
    """
    articles = read_articles_from_folder(articles_folder)
    article_ids, span_starts, span_ends, labels = read_task2_labels(template_labels_file)
    rows = []
    for aid, s, e, lab in zip(article_ids, span_starts, span_ends, labels):
        text = articles.get(aid, "")
        s = max(0, min(s, len(text)))
        e = max(0, min(e, len(text)))
        span_text = text[s:e]
        left = max(0, s - 200)
        right = min(len(text), e + 200)
        context = text[left:right].replace("\t", " ").replace("\n", " ")
        rows.append(
            {
                "article_id": aid,
                "span_start": s,
                "span_end": e,
                "span": span_text.replace("\t", " ").replace("\n", " "),
                "context": context,
                "label": lab,  # placeholder
            }
        )
    df = pd.DataFrame(rows)
    return df, articles

### 2. **Feature**
- **Tokenization**: BERT/RoBERTa tokenization with `max_seq_length=256`
- **Auxiliary Features**:
  - Length feature: span token count
  - Lexicon feature: binary indicator for Hitler-related keywords
  - Matching features: 14-d vector encoding train instance overlaps

In [ ]:
# =========================
# Data loading & preparation
# =========================

def read_articles_from_folder(folder: str) -> Dict[str, str]:
    """Read all *.txt files and return {article_id: text}.

    Expects filenames like 'article123456789.txt' but only uses the numeric suffix as id.
    """
    folder_path = Path(folder)
    articles: Dict[str, str] = {}
    for file_path in sorted(folder_path.glob("*.txt")):
        name = file_path.stem  # e.g. 'article730081389'
        # Be robust: take everything after 'article' prefix if present, else full stem
        if name.startswith("article"):
            article_id = name[len("article") :]
        else:
            article_id = name
        with file_path.open("r", encoding="utf-8") as handle:
            articles[article_id] = handle.read()
    return articles


def read_task2_labels(path: str) -> Tuple[List[str], List[int], List[int], List[str]]:
    """Read Task 2 (technique classification) labels file.

    Format per line:
        article_id <TAB> technique_label <TAB> span_start <TAB> span_end
    """
    article_ids: List[str] = []
    labels: List[str] = []
    span_starts: List[int] = []
    span_ends: List[int] = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 4:
                continue
            aid, label, start, end = parts
            article_ids.append(aid)
            labels.append(label)
            span_starts.append(int(start))
            span_ends.append(int(end))
    return article_ids, span_starts, span_ends, labels


def load_tc_data(
    articles_folder: str,
    labels_file: str,
    expand_multilabel: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Load articles and Task 2 labels and build a span-level DataFrame.

    Returns a DataFrame with:
        article_id, span_start, span_end, span_text, context, label
    and the mapping article_id -> full article text.
    """
    articles = read_articles_from_folder(articles_folder)
    article_ids, span_starts, span_ends, labels = read_task2_labels(labels_file)

    # First pass: collect all labels per unique span coordinates
    # Key: (article_id, span_start, span_end) -> set of labels
    span_to_labels: Dict[Tuple[str, int, int], set] = {}
    for aid, s, e, lab in zip(article_ids, span_starts, span_ends, labels):
        key = (aid, s, e)
        span_to_labels.setdefault(key, set()).add(lab)

    # Count multi-label spans for logging
    multi_label_count = sum(1 for labs in span_to_labels.values() if len(labs) > 1)
    if multi_label_count > 0:
        LOGGER.info(
            "Found %d multi-label spans out of %d unique spans (%.1f%%). %s",
            multi_label_count,
            len(span_to_labels),
            100.0 * multi_label_count / max(1, len(span_to_labels)),
            "Expanding to single-label rows." if expand_multilabel else "Keeping as-is."
        )
        print(
            f"Found {multi_label_count} multi-label spans out of {len(span_to_labels)} unique spans. "
            f"{'Expanding to single-label rows.' if expand_multilabel else 'Keeping as-is.'}"
        )

    rows = []
    if expand_multilabel:
        for (aid, s, e), labs in span_to_labels.items():
            text = articles.get(aid, "")
            s = max(0, min(s, len(text)))
            e = max(0, min(e, len(text)))
            span_text = text[s:e]
            context = _get_sentence_context(text, s, e)
            for lab in labs:
                rows.append(
                    {
                        "article_id": aid,
                        "span_start": s,
                        "span_end": e,
                        "span": span_text.replace("\t", " ").replace("\n", " "),
                        "context": context,
                        "label": lab,
                    }
                )
    else:
        # Original behavior: keep file order, may have duplicates
        for aid, s, e, lab in zip(article_ids, span_starts, span_ends, labels):
            text = articles.get(aid, "")
            s = max(0, min(s, len(text)))
            e = max(0, min(e, len(text)))
            span_text = text[s:e]
            context = _get_sentence_context(text, s, e)
            rows.append(
                {
                    "article_id": aid,
                    "span_start": s,
                    "span_end": e,
                    "span": span_text.replace("\t", " ").replace("\n", " "),
                    "context": context,
                    "label": lab,
                }
            )
    
    df = pd.DataFrame(rows)
    LOGGER.info("Loaded %d examples from %s", len(df), labels_file)
    return df, articles


def load_tc_test_template(
    articles_folder: str,
    template_labels_file: str,
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Load test articles and template spans.

    template_labels_file has the same format as train labels but label column is a placeholder.
    """
    articles = read_articles_from_folder(articles_folder)
    article_ids, span_starts, span_ends, labels = read_task2_labels(template_labels_file)
    rows = []
    for aid, s, e, lab in zip(article_ids, span_starts, span_ends, labels):
        text = articles.get(aid, "")
        s = max(0, min(s, len(text)))
        e = max(0, min(e, len(text)))
        span_text = text[s:e]
        context = _get_sentence_context(text, s, e)
        rows.append(
            {
                "article_id": aid,
                "span_start": s,
                "span_end": e,
                "span": span_text.replace("\t", " ").replace("\n", " "),
                "context": context,
                "label": lab,  # placeholder
            }
        )
    df = pd.DataFrame(rows)
    return df, articles



In [ ]:
# ================
# Dataset & features
# ================

@dataclass
class InputFeatures:
    input_ids: List[int]
    attention_mask: List[int]
    token_type_ids: List[int]
    label_id: Optional[int]
    length_feat: Optional[float] = None  # scalar length feature
    matchings: Optional[List[float]] = None  # 14-d vector


LABEL_ORDER = [
    'Appeal_to_Authority', 'Doubt', 'Repetition', 'Appeal_to_fear-prejudice', 'Slogans',
    'Black-and-White_Fallacy', 'Loaded_Language', 'Flag-Waving', 'Name_Calling,Labeling',
    'Whataboutism,Straw_Men,Red_Herring', 'Causal_Oversimplification', 'Exaggeration,Minimisation',
    'Bandwagon,Reductio_ad_hitlerum', 'Thought-terminating_Cliches'
]


def build_label_map(labels: Sequence[str]) -> Tuple[List[str], Dict[str, int]]:
    label_list = sorted(set(labels))
    label2id = {lab: i for i, lab in enumerate(label_list)}
    return label_list, label2id


def _simple_stem(text: str) -> str:
    # Lightweight stemmer: lowercase + strip non-alpha + collapse spaces
    tokens = re.findall(r"[A-Za-z]+", text.lower())
    return " ".join(tokens)


def build_train_instances(train_df: pd.DataFrame) -> Dict[str, set]:
    instances: Dict[str, set] = {}
    for _, r in train_df.iterrows():
        span = _simple_stem(str(r['span']))
        lab = str(r['label'])
        if span:
            instances.setdefault(span, set()).add(lab)
    return instances


def compute_matchings(span: str, train_instances: Dict[str, set]) -> List[float]:
    vec = [0.0] * len(LABEL_ORDER)
    stem = _simple_stem(span)
    labs = train_instances.get(stem, set())
    if not labs:
        return vec
    for i, name in enumerate(LABEL_ORDER):
        if name in labs:
            vec[i] = 1.0
    # normalize if any
    s = sum(vec)
    if s > 0:
        vec = [v / s for v in vec]
    return vec


def encode_examples(
    df: pd.DataFrame,
    tokenizer,
    label2id: Optional[Dict[str, int]],
    max_seq_length: int,
    is_train_or_eval: bool,
    use_length: bool = False,
    use_matchings: bool = False,
    join_embeddings: bool = False,  # kept for interface symmetry
    train_instances: Optional[Dict[str, set]] = None,
) -> Tuple[List[InputFeatures], Optional[List[str]]]:
    """Encode spans+context; optionally compute auxiliary features."""
    features: List[InputFeatures] = []
    used_labels: Optional[List[str]] = [] if is_train_or_eval else None

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Encoding examples", leave=False):
        span = str(row['span'])
        context = str(row['context'])
        encoded = tokenizer.encode_plus(
            span,
            context,
            add_special_tokens=True,
            max_length=max_seq_length,
            truncation=True,
            padding='max_length',
            return_token_type_ids=True,
        )
        input_ids = encoded['input_ids']
        attention_mask = encoded['attention_mask']
        token_type_ids = encoded.get('token_type_ids', [0] * len(input_ids))

        if is_train_or_eval:
            label_str = str(row['label'])
            label_id = label2id[label_str]
            used_labels.append(label_str)
        else:
            label_id = None

        length_feat = float(len(span.split())) if use_length else None
        matchings_vec = compute_matchings(span, train_instances) if use_matchings and train_instances else None

        features.append(InputFeatures(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            label_id=label_id,
            length_feat=length_feat,
            matchings=matchings_vec,
        ))

    return features, used_labels


def features_to_dataset(features: Sequence[InputFeatures]) -> TensorDataset:
    all_input_ids = torch.tensor([f.input_ids for f in features], dtype=torch.long)
    all_attention_mask = torch.tensor([f.attention_mask for f in features], dtype=torch.long)
    all_token_type_ids = torch.tensor([f.token_type_ids for f in features], dtype=torch.long)
    length_tensor = None
    matchings_tensor = None
    if any(f.length_feat is not None for f in features):
        length_tensor = torch.tensor([[f.length_feat or 0.0] for f in features], dtype=torch.float)
    if any(f.matchings is not None for f in features):
        matchings_tensor = torch.tensor([f.matchings or [0.0]*len(LABEL_ORDER) for f in features], dtype=torch.float)
    if features[0].label_id is not None:
        all_labels = torch.tensor([f.label_id for f in features], dtype=torch.long)
        tensors = [all_input_ids, all_attention_mask, all_token_type_ids, all_labels]
    else:
        tensors = [all_input_ids, all_attention_mask, all_token_type_ids]
    if length_tensor is not None:
        tensors.append(length_tensor)
    if matchings_tensor is not None:
        tensors.append(matchings_tensor)
    return TensorDataset(*tensors)


### 3. **Model Architecture**
- **Backbone**: RoBERTa-large (768-d hidden size)
- **Classification Head**: CLS token + pooled span embeddings + auxiliary features
- **Loss**: Weighted `CrossEntropyLoss` (inverse frequency weighting)
- **Supported variants**:
  - Base (CLS only)
  - With length features
  - With matching features
  - Joined embeddings (CLS + average pool)
  - Full ensemble (all features combined)

In [ ]:
# =========================
# Custom RoBERTa classification heads
# =========================

class RobertaClassificationHead(torch.nn.Module):
    def __init__(self, hidden_size: int, num_labels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dense = torch.nn.Linear(hidden_size, hidden_size)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, features):
        x = features[:, 0, :]
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class RobertaClassificationHeadLength(torch.nn.Module):
    def __init__(self, hidden_size: int, num_labels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dense = torch.nn.Linear(hidden_size + 1, hidden_size)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, features, sent_a_length=None):
        x = features[:, 0, :]
        x = torch.cat((x, sent_a_length), dim=1)
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class RobertaClassificationHeadMatchings(torch.nn.Module):
    def __init__(self, hidden_size: int, num_labels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dense = torch.nn.Linear(hidden_size + 14, hidden_size)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, features, matchings=None):
        x = features[:, 0, :]
        x = torch.cat((x, matchings), dim=1)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class RobertaClassificationHeadJoined(torch.nn.Module):
    def __init__(self, hidden_size: int, num_labels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dense = torch.nn.Linear(hidden_size * 2, hidden_size)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, features, attention_mask=None):
        x = features[:, 0, :]
        # mimic modeling_roberta: mask-based pooled embs over non-CLS tokens
        if attention_mask is not None:
            mask = attention_mask.reshape(features.shape[0], features.shape[1], 1).float()
            denom = mask.sum(dim=1).clamp(min=1.0)
            embs = (features * mask)[:, 1:, :].sum(dim=1) / denom
        else:
            embs = features[:, 1:, :].mean(dim=1)
        x = torch.cat((x, embs), dim=1)
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

class RobertaClassificationHeadJoinedLength(torch.nn.Module):
    def __init__(self, hidden_size: int, num_labels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dense = torch.nn.Linear(hidden_size * 2 + 1, hidden_size)
        self.dropout = torch.nn.Dropout(dropout_prob)
        self.out_proj = torch.nn.Linear(hidden_size, num_labels)


    def forward(self, features, sent_a_length=None, attention_mask=None):
        x = features[:, 0, :]
        # concat length first
        x = torch.cat((x, sent_a_length), dim=1)
        # pooled embs over non-CLS tokens (mean if no mask)
        if attention_mask is not None:
            mask = attention_mask.reshape(features.shape[0], features.shape[1], 1).float()
            embs = (features * mask)[:, 1:, :].sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        else:
            embs = features[:, 1:, :].mean(dim=1)
        x = torch.cat((x, embs), dim=1)
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class CustomRobertaForSequenceClassification(torch.nn.Module):
    def __init__(self, config: AutoConfig, use_length=False, use_matchings=False, join_embeddings=False, class_weights: Optional[torch.Tensor] = None):
        super().__init__()
        # mirror modeling_roberta: flags are in config
        config.use_length = use_length
        config.use_matchings = use_matchings
        config.join_embeddings = join_embeddings
        self.roberta = RobertaModel(config)
        self.num_labels = config.num_labels
        hidden = config.hidden_size
        dropout_prob = getattr(config, 'hidden_dropout_prob', 0.1)

        if config.use_length:
            if config.join_embeddings:
                # joined + length
                self.classifier = RobertaClassificationHeadJoinedLength(hidden, self.num_labels, dropout_prob)
            else:
                self.classifier = RobertaClassificationHeadLength(hidden, self.num_labels, dropout_prob)
        elif config.join_embeddings:
            self.classifier = RobertaClassificationHeadJoined(hidden, self.num_labels, dropout_prob)
        elif config.use_matchings:
            self.classifier = RobertaClassificationHeadMatchings(hidden, self.num_labels, dropout_prob)
        else:
            self.classifier = RobertaClassificationHead(hidden, self.num_labels, dropout_prob)

        # Optional class weights for imbalanced classes
        if class_weights is not None:
            # ensure on correct dtype; will be moved to DEVICE by caller or in forward
            self.class_weights = class_weights.float()
        else:
            self.class_weights = None
        self.loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, lengths=None, matchings=None, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        sequence_output = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.last_hidden_state
        logits = None
        # pass sequence_output and feature tensors into classifier similar to modeling_roberta
        if isinstance(self.classifier, RobertaClassificationHeadLength):
            logits = self.classifier(sequence_output, sent_a_length=lengths)
        elif isinstance(self.classifier, RobertaClassificationHeadMatchings):
            logits = self.classifier(sequence_output, matchings=matchings)
        elif isinstance(self.classifier, RobertaClassificationHeadJoinedLength):
            logits = self.classifier(sequence_output, sent_a_length=lengths, attention_mask=attention_mask)
        elif isinstance(self.classifier, RobertaClassificationHeadJoined):
            logits = self.classifier(sequence_output, attention_mask=attention_mask)
        else:
            # plain head
            logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # If weights exist but are not on the same device, move them
            if self.class_weights is not None and self.class_weights.device != logits.device:
                self.loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = self.loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        return type('Output', (), {'loss': loss, 'logits': logits})


### 4. **Training**
- AdamW optimizer with weight decay
- Linear schedule with warmup
- Early stopping based on dev F1-macro
- Gradient clipping (`max_norm=1.0`)

In [ ]:
# ===============
# Training utilities
# ===============

def set_seed(seed: int) -> None:
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_classifier(
    model,
    train_dataset: TensorDataset,
    eval_dataset: Optional[TensorDataset],
    *,
    learning_rate: float = 2e-5,
    weight_decay: float = 0.0,
    num_epochs: int = 3,
    train_batch_size: int = 16,
    eval_batch_size: int = 32,
    max_grad_norm: float = 1.0,
    warmup_ratio: float = 0.1,
    use_length: bool = False,
    use_matchings: bool = False,
    patience: int = 3,
) -> Dict[str, float]:
    """Simple training loop with optional dev evaluation and early best checkpoint in memory."""


    train_sampler = RandomSampler(train_dataset)
    train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=train_batch_size)

    t_total = len(train_loader) * max(1, num_epochs)
    warmup_steps = int(t_total * warmup_ratio)

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]

    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total
    )

    best_state = model.state_dict()
    best_eval_f1 = 0.0
    epochs_without_improvement = 0

    LOGGER.info("***** Training *****")
    print("***** Training *****")
    LOGGER.info("  Num epochs = %d", num_epochs)
    print(f"  Num epochs = {num_epochs}")
    LOGGER.info("  Train batch size = %d", train_batch_size)
    print(f"  Train batch size = {train_batch_size}")
    LOGGER.info("  Total optimization steps = %d", t_total)
    print(f"  Total optimization steps = {t_total}")
    LOGGER.info("  Early stopping patience = %d", patience)
    print(f"  Early stopping patience = {patience}")

    global_step = 0
    model.zero_grad()

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            batch = tuple(t.to(DEVICE) for t in batch)
            # base positions
            input_ids, attention_mask, token_type_ids, labels = batch[:4]
            lengths = batch[4] if use_length and len(batch) > 4 else None
            matchings = batch[5] if use_matchings and len(batch) > 5 else (batch[4] if use_matchings and len(batch) > 4 and not use_length else None)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels,
                lengths=lengths,
                matchings=matchings,
            )
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            model.zero_grad()

            global_step += 1
            epoch_loss += loss.item()

        LOGGER.info("Epoch %d | loss %.4f", epoch, epoch_loss / max(1, len(train_loader)))
        print(f"Epoch {epoch} | loss {epoch_loss / max(1, len(train_loader)):.4f}")

        if eval_dataset is not None:
            eval_metrics = evaluate_classifier(
                model, eval_dataset, batch_size=eval_batch_size,
                use_length=use_length, use_matchings=use_matchings
            )
            LOGGER.info(
                "Eval epoch %d | precision %.4f | recall %.4f | f1-macro %.4f | f1-micro %.4f",
                epoch,
                eval_metrics["precision"],
                eval_metrics["recall"],
                eval_metrics["f1_macro"],
                eval_metrics["f1_micro"],
            )
            print(f"Eval epoch {epoch} | precision {eval_metrics['precision']:.4f} | recall {eval_metrics['recall']:.4f} | f1-macro {eval_metrics['f1_macro']:.4f} | f1-micro {eval_metrics['f1_micro']:.4f}")

            # NEW: Early stopping logic
            if eval_metrics["f1_macro"] > best_eval_f1:
                best_eval_f1 = eval_metrics["f1_macro"]
                best_state = model.state_dict()
                epochs_without_improvement = 0
                LOGGER.info("New best F1: %.4f", best_eval_f1)
                print(f"✓ New best F1: {best_eval_f1:.4f}")
            else:
                epochs_without_improvement += 1
                LOGGER.info("No improvement for %d epoch(s)", epochs_without_improvement)
                print(f"No improvement for {epochs_without_improvement} epoch(s)")

                if epochs_without_improvement >= patience:
                    LOGGER.info("Early stopping triggered after %d epochs", epoch)
                    print(f"Early stopping triggered after {epoch} epochs")
                    break

    model.load_state_dict(best_state)
    return {"best_f1_macro": best_eval_f1}



def evaluate_classifier(
    model,
    dataset: TensorDataset,
    *,
    batch_size: int = 32,
    use_length: bool = False,
    use_matchings: bool = False,
) -> Dict[str, float]:
    # Local import to avoid NameError in notebook reruns
    from sklearn.metrics import precision_score, recall_score, f1_score
    sampler = SequentialSampler(dataset)
    loader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)

    model.eval()
    preds: List[int] = []
    gold: List[int] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = tuple(t.to(DEVICE) for t in batch)
            input_ids, attention_mask, token_type_ids, labels = batch[:4]
            lengths = batch[4] if use_length and len(batch) > 4 else None
            matchings = batch[5] if use_matchings and len(batch) > 5 else (batch[4] if use_matchings and len(batch) > 4 and not use_length else None)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                lengths=lengths,
                matchings=matchings,
            )
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=-1)
            preds.extend(batch_preds.cpu().tolist())
            gold.extend(labels.cpu().tolist())

    precision_macro = precision_score(gold, preds, average="macro", zero_division=0) if gold else 0.0
    recall_macro = recall_score(gold, preds, average="macro", zero_division=0) if gold else 0.0
    f1_macro = f1_score(gold, preds, average="macro") if gold else 0.0
    # Micro-F1 (same as accuracy for single-label multiclass, but added explicitly)
    f1_micro = f1_score(gold, preds, average="micro") if gold else 0.0
    return {
        "precision": float(precision_macro),
        "recall": float(recall_macro),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
    }


### 5. **Post-Processing**
- **Repetition Boost**: Detect repeated normalized spans, boost Repetition probability
- **Train-Span Boost**: Match spans against training data, boost matching labels
- **Local Consistency**: Enforce nested span compatibility rules
- **Multi-Label Recovery**: For duplicated spans, assign top-n diverse labels

In [ ]:
def predict_classifier(
    model,
    dataset: TensorDataset,
    *,
    batch_size: int = 32,
    use_length: bool = False,
    use_matchings: bool = False,
) -> np.ndarray:
    sampler = SequentialSampler(dataset)
    loader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)

    model.eval()
    all_logits: List[np.ndarray] = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting", leave=False):
            batch = tuple(t.to(DEVICE) for t in batch)
            input_ids, attention_mask, token_type_ids = batch[:3]
            # Detect whether labels are present in this dataset (dev/train have labels, test doesn't)
            has_labels = len(batch) >= 4 and batch[3].dtype == torch.long and batch[3].dim() == 1
            base_idx = 4 if has_labels else 3
            lengths = batch[base_idx] if use_length and len(batch) > base_idx else None
            matchings_idx = base_idx + 1 if use_length else base_idx
            matchings = batch[matchings_idx] if use_matchings and len(batch) > matchings_idx else None
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                lengths=lengths,
                matchings=matchings,
            )
            logits = outputs.logits
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)


### 6. **Evaluation**
- Metrics: Precision, Recall, F1-macro, F1-micro, per-class F1
- Dev and test submissions in TSV format

In [ ]:

# =================
# Submission helpers
# =================

def create_submission_file(
    test_df: pd.DataFrame,
    logits: np.ndarray,
    label_list: List[str],
    output_path: str,
) -> None:
    """Given test DataFrame, logits and label names, write submission TSV."""
    preds = np.argmax(logits, axis=1)
    with open(output_path, "w", encoding="utf-8") as handle:
        for (_, row), pred_idx in zip(test_df.iterrows(), preds):
            handle.write(
                f"{row['article_id']}\t{label_list[pred_idx]}\t{row['span_start']}\t{row['span_end']}\n"
            )


def eval_submission_file(
    submission_path: str,
    gold_labels_path: str,
    label_list: List[str],
) -> Dict[str, float]:
    # Local import to avoid NameError in notebook reruns
    from sklearn.metrics import precision_score, recall_score, f1_score
    """Compute simple accuracy and per-class F1 for a submission vs gold.

    gold_labels_path is the official labels file; we use only the label column in order.
    """
    preds: List[str] = []
    with open(submission_path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 4:
                continue
            preds.append(parts[1])

    gold: List[str] = []
    with open(gold_labels_path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 4:
                continue
            gold.append(parts[1])

    if len(preds) != len(gold):
        LOGGER.warning("Prediction/gold size mismatch: %d vs %d", len(preds), len(gold))

    # Align lengths if necessary
    n = min(len(preds), len(gold))
    preds = preds[:n]
    gold = gold[:n]

    # Map to indices for consistency
    label2id = {lab: i for i, lab in enumerate(label_list)}
    y_pred = [label2id.get(x, -1) for x in preds if x in label2id]
    y_true = [label2id.get(x, -1) for x in gold if x in label2id]

    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0
    f1_macro = f1_score(y_true, y_pred, average="macro") if y_true else 0.0
    f1_micro = f1_score(y_true, y_pred, average="micro") if y_true else 0.0
    return {
        "precision": float(precision_macro),
        "recall": float(recall_macro),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
    }


# Main function

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)
LOGGER.info("Using device: %s", DEVICE)
print(f"Using device: {DEVICE}")

model_name = "roberta-large"
max_seq_length = 256
train_batch_size = 16
eval_batch_size = 16
learning_rate = 2e-5
weight_decay = 0.01
num_epochs = 15
patience = 2
seed = 69

use_length = True
use_matchings = True
join_embeddings = True

data_root = Path("/content/drive/MyDrive/ColabNotebooks/datasets")
output_dir = Path("/content")
output_dir.mkdir(parents=True, exist_ok=True)

train_articles_dir = data_root / "train-articles"
dev_articles_dir = data_root / "dev-articles"
test_articles_dir = data_root / "test-articles"

train_labels_path = data_root / "train-task2-TC.labels"
dev_labels_path = data_root / "dev-task2-TC.labels"
test_labels_path = data_root / "test-task2-TC.labels"
set_seed(seed)

# ---- Load data ----
LOGGER.info("Loading training/dev data...")
print("Loading training/dev data...")
train_df, _ = load_tc_data(str(train_articles_dir), str(train_labels_path))
dev_df, _ = load_tc_data(str(dev_articles_dir), str(dev_labels_path))
test_df, _ = load_tc_test_template(str(test_articles_dir), str(test_labels_path))

# Optional: you can merge train+dev here and split manually
# full_df = pd.concat([train_df, dev_df], ignore_index=True)

label_list, label2id = build_label_map(train_df["label"].tolist() + dev_df["label"].tolist())
LOGGER.info("Num labels: %d", len(label_list))
print(f"Num labels: {len(label_list)}")




In [ ]:
# ---- Tokenizer & model ----
# Transfer learning support: initialize from a specified checkpoint path (file or directory)
init_model_path = Path("/content/drive/MyDrive/ColabNotebooks/models/si_model.pt")
if init_model_path.exists():
    print(f"Initializing from checkpoint: {init_model_path}")
    if init_model_path.is_dir():
        # Hugging Face-style directory with config, tokenizer, and model weights
        tokenizer = AutoTokenizer.from_pretrained(str(init_model_path))
        config = AutoConfig.from_pretrained(str(init_model_path), num_labels=len(label_list))
        base_roberta = RobertaModel.from_pretrained(str(init_model_path), config=config)
    else:
        # Single .pt file: load backbone state dict into a base model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))
        base_roberta = RobertaModel.from_pretrained(model_name, config=config)
        ckpt = torch.load(str(init_model_path), map_location="cpu")
        state_dict = ckpt.get('model_state_dict', ckpt if isinstance(ckpt, dict) else {})
        filtered = {}
        if any(k.startswith('roberta.') for k in state_dict.keys()):
            # TC-style checkpoint
            for k, v in state_dict.items():
                if k.startswith('roberta.'):
                    nk = k[len('roberta.'):]
                    filtered[nk] = v
        elif any(k.startswith('bert_encoder.') for k in state_dict.keys()):
            # SI-style checkpoint (BertLstmCrf model)
            for k, v in state_dict.items():
                if k.startswith('bert_encoder.'):
                    nk = k[len('bert_encoder.'):]
                    filtered[nk] = v
        else:
            # Assume raw RobertaModel state dict
            filtered = state_dict
        
        missing, unexpected = base_roberta.load_state_dict(filtered, strict=False)
        print(f"✓ Loaded RoBERTa weights from .pt")
        print(f"  Transferred keys: {len(filtered)}")
        print(f"  Missing keys: {len(missing)} (pooler is expected)")
        print(f"  Unexpected keys: {len(unexpected)}")
        
    model = CustomRobertaForSequenceClassification(
        config,
        use_length=use_length,
        use_matchings=use_matchings,
        join_embeddings=join_embeddings,
    ).to(DEVICE)
    model.roberta = base_roberta.to(DEVICE)
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))
    model = CustomRobertaForSequenceClassification(
        config,
        use_length=use_length,
        use_matchings=use_matchings,
        join_embeddings=join_embeddings,
    ).to(DEVICE)

In [ ]:
# Train instances for matchings
train_instances = build_train_instances(train_df) if use_matchings else None

# ---- Encode & build datasets ----
train_features, _ = encode_examples(
    train_df, tokenizer, label2id, max_seq_length, True,
    use_length=use_length, use_matchings=use_matchings, join_embeddings=join_embeddings,
    train_instances=train_instances,
)
dev_features, _ = encode_examples(
    dev_df, tokenizer, label2id, max_seq_length, True,
    use_length=use_length, use_matchings=use_matchings, join_embeddings=join_embeddings,
    train_instances=train_instances,
)
test_features, _ = encode_examples(
    test_df, tokenizer, label2id, max_seq_length, False,
    use_length=use_length, use_matchings=use_matchings, join_embeddings=join_embeddings,
    train_instances=train_instances,
)

train_dataset = features_to_dataset(train_features)
dev_dataset = features_to_dataset(dev_features)
test_dataset = features_to_dataset(test_features)

# ---- Train ----
model_save_path = output_dir / "tc_model.pt"
tokenizer_save_path = output_dir / "tokenizer"
LOGGER.info("Starting training...")
print("Starting training...")
train_metrics = train_classifier(
    model,
    train_dataset,
    dev_dataset,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    num_epochs=num_epochs,
    train_batch_size=train_batch_size,
    eval_batch_size=eval_batch_size,
    use_length=use_length,
    use_matchings=use_matchings,
    patience=patience,
)


LOGGER.info("Best dev F1-macro: %.4f", train_metrics["best_f1_macro"])
print(f"Best dev F1-macro: {train_metrics['best_f1_macro']:.4f}")

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
    'label_list': label_list,
    'use_length': use_length,
    'use_matchings': use_matchings,
    'join_embeddings': join_embeddings,
}, model_save_path)
LOGGER.info("Model saved to %s", model_save_path)
print(f"Model saved to {model_save_path}")

# Save tokenizer separately
tokenizer_save_path.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(tokenizer_save_path)
LOGGER.info("Tokenizer saved to %s", tokenizer_save_path)
print(f"Tokenizer saved to {tokenizer_save_path}")
# ---- Predict on dev & test ----
LOGGER.info("Predicting on dev & test...")
print("Predicting on dev & test...")
dev_logits = predict_classifier(
    model, dev_dataset, batch_size=eval_batch_size,
    use_length=use_length, use_matchings=use_matchings,
)
test_logits = predict_classifier(
    model, test_dataset, batch_size=eval_batch_size,
    use_length=use_length, use_matchings=use_matchings,
)

dev_submission_path = output_dir / "dev_submission_task2.tsv"
test_submission_path = output_dir / "test_submission_task2.tsv"
create_submission_file(dev_df, dev_logits, label_list, str(dev_submission_path))
create_submission_file(test_df, test_logits, label_list, str(test_submission_path))

# ---- Evaluate on dev using gold labels ----
dev_scores = eval_submission_file(str(dev_submission_path), str(dev_labels_path), label_list)
LOGGER.info(
    "Dev submission metrics - precision: %.4f | recall: %.4f | f1-macro: %.4f | f1-micro:%.4f",
    dev_scores["precision"],
    dev_scores["recall"],
    dev_scores["f1_macro"],
    dev_scores["f1_micro"],
)
test_scores = eval_submission_file(str(test_submission_path), str(test_labels_path), label_list)
LOGGER.info(
    "Test submission metrics (vs dev gold) - precision: %.4f | recall: %.4f | f1-macro: %.4f | f1-micro:%.4f",
    test_scores["precision"],
    test_scores["recall"],
    test_scores["f1_macro"],
    test_scores["f1_micro"],
)
print(f"Dev submission metrics - precision: {dev_scores['precision']:.4f} | recall: {dev_scores['recall']:.4f} | f1-macro: {dev_scores['f1_macro']:.4f}| f1-micro:{dev_scores['f1_micro']}")
print(f"Test submission metrics - precision: {test_scores['precision']:.4f} | recall: {test_scores['recall']:.4f} | f1-macro: {test_scores['f1_macro']:.4f}|f1-micro:{test_scores['f1_micro']}")
LOGGER.info("Dev submission written to %s", dev_submission_path)
print(f"Dev submission written to {dev_submission_path}")
LOGGER.info("Test submission written to %s", test_submission_path)
print(f"Test submission written to {test_submission_path}")